# Очистка и подготовка данных

## Импорты и загрузка

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "rent_offers_raw.csv"
PROCESSED_CSV = PROJECT_ROOT / "data" / "processed" / "rent_offers_clean.csv"

df = pd.read_csv(RAW_CSV)
print(f"Загружено строк: {df.shape[0]}, колонок: {df.shape[1]}")
print("Колонки:", list(df.columns))

Загружено строк: 5037, колонок: 17
Колонки: ['author', 'author_type', 'url', 'location', 'deal_type', 'accommodation_type', 'floor', 'floors_count', 'rooms_count', 'total_meters', 'price_per_month', 'commissions', 'district', 'street', 'house_number', 'underground', 'residential_complex']


## Первичный осмотр

In [2]:
print("Типы данных:")
print(df.dtypes)
print("\nПропуски по колонкам (в %):")
print((df.isnull().mean() * 100).round(1).sort_values(ascending=False))
print("\nПервые 3 строки:")
df.head(3)

Типы данных:
author                     str
author_type                str
url                        str
location                   str
deal_type                  str
accommodation_type         str
floor                    int64
floors_count             int64
rooms_count              int64
total_meters           float64
price_per_month          int64
commissions              int64
district                   str
street                     str
house_number               str
underground                str
residential_complex        str
dtype: object

Пропуски по колонкам (в %):
residential_complex    96.9
district                6.5
street                  0.2
house_number            0.1
author                  0.0
author_type             0.0
url                     0.0
location                0.0
deal_type               0.0
rooms_count             0.0
floors_count            0.0
floor                   0.0
accommodation_type      0.0
commissions             0.0
price_per_month         0

,author,author_type,url,location,deal_type,accommodation_type,floor,floors_count,rooms_count,total_meters,price_per_month,commissions,district,street,house_number,underground,residential_complex
0,ЛюксДом,real_estate_agent,https://www.cian.ru/rent/flat/333623858/?conte...,Москва,rent,flat,8,12,3,158.4,1000000,0,Хамовники,Ефремова,10к2,Спортивная,NaN
1,ID 22391002,homeowner,https://www.cian.ru/rent/flat/291773938/?conte...,Москва,rent,flat,2,15,1,32.0,100000,0,Хорошевский,Ходынский бульвар,20А,ЦСКА,NaN
2,ID 29054506,official_representative,https://www.cian.ru/rent/flat/334877071/?conte...,Москва,rent,flat,15,33,2,35.1,70000,0,Хорошево-Мневники,Причальный проезд,10к1,Шелепиха,NaN


## Устойчивый поиск нужных колонок

In [3]:
COLUMN_CANDIDATES = {
    "price":               ["price_per_month"],
    "area":                ["total_meters"],
    "rooms":               ["rooms_count"],
    "floor":               ["floor"],
    "floors_total":        ["floors_count"],
    "district":            ["district"],
    "url":                 ["url"],
    "underground":         ["underground"],
    "residential_complex": ["residential_complex"],
    "author_type":         ["author_type"],
    # street и house_number оставляем как есть, склеим в address ниже
}

rename_map = {}
found = {}
for target, candidates in COLUMN_CANDIDATES.items():
    for cand in candidates:
        if cand in df.columns:
            rename_map[cand] = target
            found[target] = cand
            break
    else:
        found[target] = None

print("Найдено соответствие колонок:")
for target, source in found.items():
    status = f"'{source}'" if source else "❌ НЕ НАЙДЕНА"
    print(f"  {target:22s} ← {status}")

df = df.rename(columns=rename_map)

Найдено соответствие колонок:
  price                  ← 'price_per_month'
  area                   ← 'total_meters'
  rooms                  ← 'rooms_count'
  floor                  ← 'floor'
  floors_total           ← 'floors_count'
  district               ← 'district'
  url                    ← 'url'
  underground            ← 'underground'
  residential_complex    ← 'residential_complex'
  author_type            ← 'author_type'


In [4]:
def build_address(row):
    parts = [row.get("street"), row.get("house_number")]
    parts = [str(p).strip() for p in parts if pd.notna(p) and str(p).strip()]
    return ", ".join(parts) if parts else None

df["address"] = df.apply(build_address, axis=1)

print(df[["district", "street", "house_number", "address"]].head(5))

            district                  street house_number  \
0          Хамовники                Ефремова         10к2   
1        Хорошевский       Ходынский бульвар          20А   
2  Хорошево-Мневники       Причальный проезд         10к1   
3           Аэропорт  Ленинградский проспект        36с38   
4      Академический     Новочеремушкинская            17   

                         address  
0                 Ефремова, 10к2  
1         Ходынский бульвар, 20А  
2        Причальный проезд, 10к1  
3  Ленинградский проспект, 36с38  
4         Новочеремушкинская, 17  


## Приведение чисел к числовому типу

In [5]:
def to_number(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)
    s = str(value).replace("\xa0", "").replace(" ", "").replace(",", ".")
    m = re.search(r"-?\d+\.?\d*", s)
    return float(m.group()) if m else np.nan


for col in ["price", "area", "floor", "floors_total", "rooms"]:
    if col in df.columns:
        df[col] = df[col].apply(to_number)

print("После приведения к числам:")
print(df[["price", "area", "floor", "floors_total", "rooms"]].describe().round(1))

После приведения к числам:
           price    area   floor  floors_total   rooms
count     5037.0  5037.0  5037.0        5037.0  5037.0
mean    312465.6    86.0     9.7          18.9     1.9
std     421072.0    82.9     8.8          15.1     1.4
min      30000.0    11.0     1.0           2.0    -1.0
25%      77900.0    39.0     3.0           9.0     1.0
50%     120000.0    49.0     8.0          15.0     2.0
75%     349000.0   100.0    13.0          22.0     3.0
max    3500000.0   700.0    71.0          97.0     5.0


## Отсечение пропусков и явных выбросов

In [6]:
rows_before = len(df)

# Обязательные поля: без цены или площади объявление бесполезно
df = df.dropna(subset=["price", "area"])

# Правдоподобные диапазоны для Москвы
df = df[
    (df["price"].between(20_000, 1_000_000)) &   # 20к–1 млн ₽/мес
    (df["area"].between(10, 400))                # 10–400 м²
]

# Комнаты: 0 — студия, максимум 10 (в Москве редкость, но бывает)
if "rooms" in df.columns:
    df = df[df["rooms"].between(0, 10) | df["rooms"].isna()]

print(f"Строк до фильтрации: {rows_before}")
print(f"После фильтрации: {len(df)} (удалено {rows_before - len(df)})")

Строк до фильтрации: 5037
После фильтрации: 4291 (удалено 746)


## Новые признаки: цена за кв.м и «этажность» квартиры

In [8]:
df["price_per_sqm"] = (df["price"] / df["area"]).round(0)

# Первый/последний этаж — классический фактор цены в аренде
df["is_first_floor"] = (df["floor"] == 1).astype(int)
df["is_last_floor"] = (
    (df["floor"].notna())
    & (df["floors_total"].notna())
    & (df["floor"] == df["floors_total"])
).astype(int)

# Доля этажа в доме (0 = первый, 1 = последний) — иногда информативнее, чем просто номер
df["floor_ratio"] = (df["floor"] / df["floors_total"]).round(2)

print(df[["price", "area", "price_per_sqm", "floor", "floors_total", "floor_ratio"]].head())
print("\nВыбросы по цене за кв.м (describe):")
print(df["price_per_sqm"].describe().round(0))

       price   area  price_per_sqm  floor  floors_total  floor_ratio
0  1000000.0  158.4         6313.0    8.0          12.0         0.67
1   100000.0   32.0         3125.0    2.0          15.0         0.13
2    70000.0   35.1         1994.0   15.0          33.0         0.45
3   280000.0   48.7         5749.0   13.0          15.0         0.87
4   105000.0   42.0         2500.0    3.0          22.0         0.14

Выбросы по цене за кв.м (describe):
count     4291.0
mean      2825.0
std       1427.0
min        304.0
25%       1667.0
50%       2564.0
75%       3500.0
max      11250.0
Name: price_per_sqm, dtype: float64


## Финальная чистка выбросов по цене за кв.м

In [9]:
rows_before = len(df)
df = df[df["price_per_sqm"].between(300, 5000)]
print(f"Удалено выбросов по цене за кв.м: {rows_before - len(df)}")
print(f"Итог: {len(df)} строк")

Удалено выбросов по цене за кв.м: 324
Итог: 3967 строк


## Комнатность: приводим студии и категории

In [10]:
if "rooms" in df.columns:
    df["rooms"] = df["rooms"].fillna(-1).astype(int)
    df["rooms_label"] = df["rooms"].map({
        0: "Студия", 1: "1-комн.", 2: "2-комн.",
        3: "3-комн.", 4: "4-комн.", 5: "5+ комн.", -1: "Не указано",
    }).fillna("5+ комн.")
    print(df["rooms_label"].value_counts())

rooms_label
2-комн.     1712
1-комн.     1195
3-комн.      715
4-комн.      306
5+ комн.      39
Name: count, dtype: int64


## Финальный чек и сохранение

In [ ]:
print("=" * 60)
print(f"Итоговый датасет: {df.shape[0]} строк, {df.shape[1]} колонок")
print("\nПропуски по ключевым полям (в %):")
key_cols = [
    c for c in [
        "price", "area", "price_per_sqm", "rooms", "district",
        "floor", "author_type", "residential_complex", "underground",
    ] if c in df.columns
]
print((df[key_cols].isnull().mean() * 100).round(1))
print("\nКолонки в финале:", list(df.columns))
print("=" * 60)

PROCESSED_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_CSV, index=False, encoding="utf-8-sig")
print(f"\n✅ Сохранено: {PROCESSED_CSV.relative_to(PROJECT_ROOT)}")

Итоговый датасет: 3967 строк, 23 колонок

Пропуски по ключевым полям (в %):
price                   0.0
area                    0.0
price_per_sqm           0.0
rooms                   0.0
district                8.2
floor                   0.0
author_type             0.1
residential_complex    96.0
underground             0.0
dtype: float64

Колонки в финале: ['author', 'author_type', 'url', 'location', 'deal_type', 'accommodation_type', 'floor', 'floors_total', 'rooms', 'area', 'price', 'commissions', 'district', 'street', 'house_number', 'underground', 'residential_complex', 'address', 'price_per_sqm', 'is_first_floor', 'is_last_floor', 'floor_ratio', 'rooms_label']

✅ Сохранено: data\processed\rent_offers_clean.csv
